# 04 — Simulated Policy Portfolio (Phase 4)

โน้ตบุ๊กนี้เป็น narrative shell สำหรับ Phase 4: import และเรียกใช้ฟังก์ชันจาก `src/rri/warehouse/`,
`src/rri/features/`, และ `config/simulation_params.py` เท่านั้น — ไม่มี business logic ในเซลล์
ตาม CLAUDE.md N2

**ทำไมต้องมี portfolio จำลอง:** SiamShield เป็นบริษัทประกันสมมติ เราไม่มีสมุดกรมธรรม์จริง
พอร์ตนี้มีไว้เพื่อแปลงความเสี่ยงจริง (`fact_accident`) ให้อยู่ในรูปแบบธุรกิจประกัน (loss ratio,
premium) — **ML ใน Phase 5-6 เทรนจาก `fact_accident` เท่านั้น ไม่ใช้พอร์ตจำลองนี้** เพื่อไม่ให้
โมเดลย้อนกลับไปค้นพบสมมติฐานที่เราใส่เข้าไปเอง (circular reasoning, CLAUDE.md §7.3)

ลำดับขั้นตอน:

1. ค่าคงที่อ้างอิง (`config/simulation_params.py`, Option B — ช่วงตัวเลขที่เผยแพร่จริง ไม่ใช่
   ตัวเลขของบริษัทใดบริษัทหนึ่ง)
2. Populate `dim_policy_type` / `dim_customer_segment`
3. สร้าง `fact_underwriting_monthly` (exposure back-cast + claim derivation)
4. **CHECKPOINT** — business sanity gate (loss ratio 55-75%), semi-additive check
5. Export ทั้งหมดไปยัง `data/40_exports/` + ER diagram


## 0. Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import duckdb

from config.simulation_params import (
    ANNUAL_PREMIUM_BY_COVER_CLASS_THB,
    CLAIM_FREQUENCY,
    CLAIM_FREQUENCY_RANGE,
    COVER_CLASS_MIX,
    MEAN_CLAIM_SEVERITY_RANGE_THB,
    MEAN_CLAIM_SEVERITY_THB,
    SEVERITY_WEIGHT_FATAL,
    SEVERITY_WEIGHT_MINOR,
    SEVERITY_WEIGHT_SERIOUS,
)
from rri.utils.logging import configure_logging
from rri.warehouse.build import build_dimensions, load_staging_frames
from rri.warehouse.dim_vehicle import vehicle_group_mapping_path
from rri.warehouse.duckdb_io import get_connection, write_table
from rri.warehouse.fact_accident import build_fact_accident
from rri.warehouse.fact_underwriting_monthly import (
    MEASURE_ADDITIVITY,
    WINDOW_END,
    WINDOW_START,
    build_fact_underwriting_monthly,
)

configure_logging()

'422fb117b1524536b3c4b81c497d5c45'

## 1. ค่าคงที่อ้างอิง (Option B)

ทุกตัวเลขมี one-line sourcing note ใน `config/simulation_params.py` — รายละเอียดเต็มอยู่ที่
`docs/simulation_assumptions.md`

In [2]:
freq_lo, freq_hi = CLAIM_FREQUENCY_RANGE
sev_lo, sev_hi = MEAN_CLAIM_SEVERITY_RANGE_THB
print(f"Claim frequency: {CLAIM_FREQUENCY:.0%} (published range {freq_lo:.0%}-{freq_hi:.0%})")
print(
    f"Mean claim severity: {MEAN_CLAIM_SEVERITY_THB:,.0f} THB "
    f"(published range {sev_lo:,.0f}-{sev_hi:,.0f})"
)
print(
    f"severity_index_fine weights (fatal:serious:minor): "
    f"{SEVERITY_WEIGHT_FATAL:.0f}:{SEVERITY_WEIGHT_SERIOUS:.0f}:{SEVERITY_WEIGHT_MINOR:.0f}"
)
print(f"Cover-class mix: {COVER_CLASS_MIX}")
print(f"Annual premium by class (THB): {ANNUAL_PREMIUM_BY_COVER_CLASS_THB}")

Claim frequency: 6% (published range 5%-8%)
Mean claim severity: 30,000 THB (published range 20,000-40,000)
severity_index_fine weights (fatal:serious:minor): 40:8:1
Cover-class mix: {'class_1': 0.25, 'class_2plus': 0.1, 'class_3plus': 0.15, 'class_3': 0.5}
Annual premium by class (THB): {'class_1': 9000.0, 'class_2plus': 4400.0, 'class_3plus': 2900.0, 'class_3': 1800.0}


## 2. โหลด staging + สร้าง dimensions (populate `dim_policy_type`/`dim_customer_segment` จริง)

`build_dimensions` เรียก `build_dim_policy_type`/`build_dim_customer_segment` พร้อมสมาชิกจริง
จาก `config/simulation_params.py` (cover-class mix, age/gender/customer-type) — แทนที่ scaffold
เปล่าจาก Phase 3 (notebook 03) ด้วยสมาชิกจริง

In [3]:
staging = load_staging_frames()
dims = build_dimensions(staging)
print("dim_policy_type:", dims["dim_policy_type"].height, "rows")
print("dim_customer_segment:", dims["dim_customer_segment"].height, "rows")
dims["dim_policy_type"]

dim_policy_type: 5 rows
dim_customer_segment: 21 rows


policy_type_sk,cover_class,cover_class_label,is_comprehensive
i64,str,str,bool
-1,"""unknown""","""Unknown""",false
1,"""class_1""","""Class 1 (Comprehensive)""",true
2,"""class_2plus""","""Class 2+ (Fire/Theft + Collisi…",false
3,"""class_3plus""","""Class 3+ (Collision only)""",false
4,"""class_3""","""Class 3 (Third-Party Only)""",false


## สร้าง `fact_accident` ใหม่ (dims เพิ่งสร้างใหม่ในเซสชันนี้ ต้องสอดคล้องกัน)

In [4]:
fact_accident = build_fact_accident(
    staging["accidents"],
    dims["dim_date"],
    dims["dim_geography"],
    dims["dim_vehicle"],
    dims["dim_accident_context"],
    dims["dim_time_of_day"],
    dims["dim_weather_band"],
    vehicle_group_mapping_path(),
)
fact_accident.height

159605

## 3. สร้าง `fact_underwriting_monthly`

Grain: หนึ่งเดือน × หนึ่งจังหวัด × หนึ่งกลุ่มรถ × หนึ่ง cover class

**หน้าต่างเวลา: 2022-01 ถึง 2025-12 (48 เดือน)** ไม่ใช่ประมาณการเดิมของ CLAUDE.md §8
("~92,400 แถว" ซึ่งสมมติ 60 เดือน) — เพราะ exposure back-cast (§7.1) ต้องใช้ national growth
curve จาก `car_registered_new` ซึ่งมีข้อมูลตั้งแต่ 2022-01 เท่านั้น การยืดย้อนกลับไปถึง 2019
จะเป็นการสร้างข้อมูล growth-rate ที่ source ไม่มีอยู่จริงขึ้นมาเอง

**`claim_count` derivation (แก้ไขระหว่าง development):** เดิมออกแบบให้ `claim_count`
มาจาก raw accident count ต่อ cell โดยตรง × `CLAIM_FREQUENCY` — พบว่า loss ratio ออกมา
~0.07% (ต่ำกว่าเป้าหมาย 55-75% ถึง 3 order of magnitude) เพราะ accident count ต่อ cell
เบาบางเกินไป และไม่สัมพันธ์กับ scale ของ `policies_in_force` (มาจาก exposure จริงซึ่งมีขนาด
หลักล้านคันในจังหวัดใหญ่) แก้เป็นความสัมพันธ์ actuarial มาตรฐาน:
`claim_count = policies_in_force × (CLAIM_FREQUENCY / 12)` แทน — รายละเอียดเต็มอยู่ใน
module docstring ของ `fact_underwriting_monthly.py`

In [5]:
fact_uw = build_fact_underwriting_monthly(
    fact_accident,
    dims["dim_date"],
    dims["dim_geography"],
    dims["dim_vehicle"],
    dims["dim_policy_type"],
    staging["stt_car_fuel"],
    staging["car_registered_new"],
    vehicle_group_mapping_path(),
)
print("window:", WINDOW_START, "-", WINDOW_END)
print("rows:", fact_uw.height)
fact_uw.head(3)

window: (2022, 1) - (2025, 12)
rows: 73920


underwriting_sk,date_sk,geography_sk,vehicle_sk,policy_type_sk,policies_in_force,exposure_vehicle_months,earned_premium,claim_count,incurred_claim_amount,paid_claim_amount
i64,i64,i64,i64,i64,i64,f64,f64,i64,f64,f64
1,20220101,1,1,1,970,23888.39,727500.0,5,136523.478865,116044.957035
2,20220101,1,1,2,362,9555.36,132733.333333,2,54609.391546,46417.982814
3,20220101,1,1,3,595,14333.03,143791.666667,3,81914.087319,69626.974221


## 4. เขียนลง `warehouse.duckdb` (dims ทั้ง 8 + fact ทั้ง 2)

In [6]:
con = get_connection()
try:
    for table_name, df in dims.items():
        write_table(con, table_name, df)
    write_table(con, "fact_accident", fact_accident)
    write_table(con, "fact_underwriting_monthly", fact_uw)
finally:
    con.close()
print("written: 8 dims + fact_accident + fact_underwriting_monthly")

2026-08-14T07:19:36.094561Z [info     ] wrote_table                    cols=21 rows=2923 run_id=422fb117b1524536b3c4b81c497d5c45 table=dim_date


2026-08-14T07:19:36.096652Z [info     ] wrote_table                    cols=15 rows=78 run_id=422fb117b1524536b3c4b81c497d5c45 table=dim_geography


2026-08-14T07:19:36.097845Z [info     ] wrote_table                    cols=4 rows=6 run_id=422fb117b1524536b3c4b81c497d5c45 table=dim_vehicle


2026-08-14T07:19:36.099363Z [info     ] wrote_table                    cols=5 rows=1679 run_id=422fb117b1524536b3c4b81c497d5c45 table=dim_accident_context


2026-08-14T07:19:36.100456Z [info     ] wrote_table                    cols=4 rows=25 run_id=422fb117b1524536b3c4b81c497d5c45 table=dim_time_of_day


2026-08-14T07:19:36.101315Z [info     ] wrote_table                    cols=3 rows=6 run_id=422fb117b1524536b3c4b81c497d5c45 table=dim_weather_band


2026-08-14T07:19:36.102273Z [info     ] wrote_table                    cols=4 rows=5 run_id=422fb117b1524536b3c4b81c497d5c45 table=dim_policy_type


2026-08-14T07:19:36.103187Z [info     ] wrote_table                    cols=4 rows=21 run_id=422fb117b1524536b3c4b81c497d5c45 table=dim_customer_segment


2026-08-14T07:19:36.215970Z [info     ] wrote_table                    cols=24 rows=159605 run_id=422fb117b1524536b3c4b81c497d5c45 table=fact_accident


2026-08-14T07:19:36.225287Z [info     ] wrote_table                    cols=11 rows=73920 run_id=422fb117b1524536b3c4b81c497d5c45 table=fact_underwriting_monthly


written: 8 dims + fact_accident + fact_underwriting_monthly


## 5. CHECKPOINT — Business Sanity Gate

Overall loss ratio ต้องอยู่ในช่วง 55-75% — ถ้าออกมา 300% หรือ 5% แปลว่าค่าคงที่ผิด
ต้องหยุดตรวจ ไม่ใช่ปล่อยผ่าน

In [7]:
warehouse_path = Path.cwd().parent / "data" / "30_warehouse" / "warehouse.duckdb"
con = duckdb.connect(str(warehouse_path), read_only=True)

loss_ratio = con.sql(
    "SELECT SUM(incurred_claim_amount) / SUM(earned_premium) AS loss_ratio "
    "FROM fact_underwriting_monthly"
).fetchone()[0]
print(f"Overall loss ratio: {loss_ratio:.1%}")
assert 0.55 <= loss_ratio <= 0.75, f"loss ratio {loss_ratio:.1%} outside the 55-75% sanity band"

Overall loss ratio: 65.3%


### Semi-additive check

`policies_in_force` ห้าม sum ข้ามเดือน (เป็น snapshot ไม่ใช่ flow) — `MEASURE_ADDITIVITY`
บันทึกกฎนี้ไว้ ใครจะ query ต้องเช็คก่อน

In [8]:
for measure, additivity in MEASURE_ADDITIVITY.items():
    print(f"  {measure}: {additivity}")
assert MEASURE_ADDITIVITY["policies_in_force"].startswith("semi-additive")

  policies_in_force: semi-additive (never sum across months)
  exposure_vehicle_months: additive
  earned_premium: additive
  claim_count: additive
  incurred_claim_amount: additive
  paid_claim_amount: additive


### Orphan FK check (fact_underwriting_monthly)

In [9]:
fks = {
    "date_sk": ("dim_date", "date_sk"),
    "geography_sk": ("dim_geography", "geography_sk"),
    "vehicle_sk": ("dim_vehicle", "vehicle_sk"),
    "policy_type_sk": ("dim_policy_type", "policy_type_sk"),
}
total_orphans = 0
for fk, (dim, dimkey) in fks.items():
    n = con.sql(
        f"SELECT COUNT(*) FROM fact_underwriting_monthly f "
        f"LEFT JOIN {dim} d ON f.{fk} = d.{dimkey} "
        f"WHERE d.{dimkey} IS NULL"
    ).fetchone()[0]
    total_orphans += n
    print(f"  {fk}: {n} orphans")
assert total_orphans == 0
con.close()

  date_sk: 0 orphans
  geography_sk: 0 orphans
  vehicle_sk: 0 orphans
  policy_type_sk: 0 orphans


**สรุป checkpoint:** loss ratio อยู่ในช่วงเป้าหมาย, semi-additive flag ถูกบันทึกและตรวจสอบแล้ว,
ไม่มี orphan FK → พอร์ตจำลองพร้อมใช้งาน

## 6. Export ทั้งหมด + ER Diagram

In [10]:
from scripts.generate_er_diagram import generate_er_diagram_markdown

from rri.warehouse.export import TABLE_PRIMARY_KEYS, export_all

con = get_connection()
try:
    export_all(con, Path.cwd().parent / "data" / "40_exports", list(TABLE_PRIMARY_KEYS.keys()))
    er_markdown = generate_er_diagram_markdown(con)
finally:
    con.close()

er_path = Path.cwd().parent / "reports" / "er_diagram.md"
er_path.write_text(er_markdown, encoding="utf-8")
print("exported all tables (Parquet + CSV + schema_mysql.sql) and wrote", er_path)

2026-08-14T07:19:36.271745Z [info     ] exported_table                 csv='/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/40_exports/dim_date.csv' parquet='/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/40_exports/dim_date.parquet' run_id=422fb117b1524536b3c4b81c497d5c45 table=dim_date


2026-08-14T07:19:36.273132Z [info     ] exported_table                 csv='/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/40_exports/dim_geography.csv' parquet='/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/40_exports/dim_geography.parquet' run_id=422fb117b1524536b3c4b81c497d5c45 table=dim_geography


2026-08-14T07:19:36.274008Z [info     ] exported_table                 csv='/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/40_exports/dim_vehicle.csv' parquet='/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/40_exports/dim_vehicle.parquet' run_id=422fb117b1524536b3c4b81c497d5c45 table=dim_vehicle


2026-08-14T07:19:36.275673Z [info     ] exported_table                 csv='/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/40_exports/dim_accident_context.csv' parquet='/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/40_exports/dim_accident_context.parquet' run_id=422fb117b1524536b3c4b81c497d5c45 table=dim_accident_context


2026-08-14T07:19:36.276644Z [info     ] exported_table                 csv='/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/40_exports/dim_time_of_day.csv' parquet='/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/40_exports/dim_time_of_day.parquet' run_id=422fb117b1524536b3c4b81c497d5c45 table=dim_time_of_day


2026-08-14T07:19:36.277527Z [info     ] exported_table                 csv='/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/40_exports/dim_weather_band.csv' parquet='/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/40_exports/dim_weather_band.parquet' run_id=422fb117b1524536b3c4b81c497d5c45 table=dim_weather_band


... (2 outputs left out)

2026-08-14T07:19:36.394623Z [info     ] exported_table                 csv='/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/40_exports/fact_accident.csv' parquet='/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/40_exports/fact_accident.parquet' run_id=422fb117b1524536b3c4b81c497d5c45 table=fact_accident


2026-08-14T07:19:36.434797Z [info     ] exported_table                 csv='/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/40_exports/fact_underwriting_monthly.csv' parquet='/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/40_exports/fact_underwriting_monthly.parquet' run_id=422fb117b1524536b3c4b81c497d5c45 table=fact_underwriting_monthly


2026-08-14T07:19:36.438074Z [info     ] wrote_mysql_schema             path='/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/40_exports/schema_mysql.sql' run_id=422fb117b1524536b3c4b81c497d5c45


exported all tables (Parquet + CSV + schema_mysql.sql) and wrote /Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/reports/er_diagram.md
